# 03 - Evaluation

Runs all evaluation scripts (metrics, stats, scaling, convergence,
per-image, qualitative) and then generates all figures for the thesis.

In [ ]:
# Zelle 0 - Imports
import csv
import shutil
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
# Zelle 1 - Setup + Drive
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/ba-mamba-neck'
os.chdir(REPO)
sys.path.insert(0, REPO)

# Copy seed result CSVs from Drive to local results/
DRIVE_BASE = Path('/content/drive/MyDrive/ba')
results = Path('results')
results.mkdir(exist_ok=True)

for neck in ('fpn', 'aifi', 'mamba'):
    for csv_name in (f'{neck}_seed_results.csv', f'{neck}_convergence.csv'):
        src = results / csv_name
        if src.exists():
            print(f'{csv_name} already present')
        else:
            print(f'WARNING: {csv_name} not found in results/ - '
                  f'run notebook 02 for {neck} first')

In [ ]:
# Zelle 2 - Metrics: merge CSVs, summary, deltas
!PYTHONPATH=. python eval/metrics.py

In [ ]:
# Zelle 3 - Stats: Friedman, Nemenyi, Wilcoxon, CD diagrams
!PYTHONPATH=. python eval/stats.py

In [ ]:
# Zelle 4 - Scaling: latency, memory, GFLOPs vs resolution
!PYTHONPATH=. python eval/scaling.py

In [ ]:
# Zelle 5 - Convergence: merge + summary CSVs
!PYTHONPATH=. python eval/convergence.py

In [ ]:
# Zelle 6 - Per-image analysis + Mamba-vs-FPN divergence
!PYTHONPATH=. python eval/per_image.py

In [ ]:
# Zelle 7 - Qualitative visualisation (top-10 divergence images)
!PYTHONPATH=. python eval/qualitative.py

In [ ]:
# Zelle 8 - ALL FIGURES for thesis
sns.set_theme(context='paper', style='whitegrid', font_scale=1.2)

FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {'fpn': '#2196F3', 'aifi': '#9C27B0', 'mamba': '#009688'}
NECKS = ['fpn', 'aifi', 'mamba']
NECK_LABELS = {'fpn': 'FPN (V1)', 'aifi': 'AIFI (V2)', 'mamba': 'Mamba (V3)'}


def _read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


# --- Helper: save PDF ---
def _save(fig, name):
    p = FIG_DIR / f'{name}.pdf'
    fig.savefig(p, bbox_inches='tight')
    plt.close(fig)
    print(f'  {p}')


# --- 1. AP by size (grouped bar with error bars) ---
summary = _read_csv('results/detection_summary.csv')
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(3)  # S, M, L
w = 0.25
for i, neck in enumerate(NECKS):
    row = [r for r in summary if r['neck'] == neck][0]
    means = [float(row[f'AP_{s}_mean']) for s in ('S', 'M', 'L')]
    stds = [float(row[f'AP_{s}_std']) for s in ('S', 'M', 'L')]
    ax.bar(x + i * w, means, w, yerr=stds, label=NECK_LABELS[neck],
           color=COLORS[neck], capsize=3)
ax.set_xticks(x + w)
ax.set_xticklabels(['AP_S', 'AP_M', 'AP_L'])
ax.set_ylabel('Average Precision')
ax.legend()
ax.set_title('Detection AP by Object Size')
_save(fig, 'ap_by_size')

# --- 2. Latency vs resolution ---
scaling = _read_csv('results/scaling.csv')
fig, ax = plt.subplots(figsize=(6, 4))
for neck in NECKS:
    rows = [r for r in scaling if r['neck'] == neck]
    resolutions = [int(r['resolution']) for r in rows]
    latencies = [float(r['latency_ms']) for r in rows]
    ax.plot(resolutions, latencies, 'o-', color=COLORS[neck],
            label=NECK_LABELS[neck])
ax.set_xlabel('Input Resolution (px)')
ax.set_ylabel('Latency (ms/image)')
ax.legend()
ax.set_title('Inference Latency vs Resolution')
_save(fig, 'latency_vs_resolution')

# --- 3. GFLOPs vs resolution ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
for neck in NECKS:
    rows = [r for r in scaling if r['neck'] == neck]
    res = [int(r['resolution']) for r in rows]
    ax1.plot(res, [float(r['gflops_neck']) for r in rows], 'o-',
             color=COLORS[neck], label=NECK_LABELS[neck])
    ax2.plot(res, [float(r['gflops_total']) for r in rows], 'o-',
             color=COLORS[neck], label=NECK_LABELS[neck])
ax1.set_title('Neck-only GFLOPs'); ax1.set_xlabel('Resolution'); ax1.set_ylabel('GFLOPs')
ax2.set_title('Full Model GFLOPs'); ax2.set_xlabel('Resolution'); ax2.set_ylabel('GFLOPs')
ax1.legend(); ax2.legend()
_save(fig, 'gflops_vs_resolution')

# --- 4. Memory vs resolution ---
fig, ax = plt.subplots(figsize=(6, 4))
for neck in NECKS:
    rows = [r for r in scaling if r['neck'] == neck]
    res = [int(r['resolution']) for r in rows]
    mem = [float(r['memory_gb']) for r in rows]
    ax.plot(res, mem, 'o-', color=COLORS[neck], label=NECK_LABELS[neck])
ax.set_xlabel('Resolution'); ax.set_ylabel('Peak GPU Memory (GB)')
ax.legend(); ax.set_title('GPU Memory vs Resolution')
_save(fig, 'memory_vs_resolution')

# --- 5. CD diagrams (already generated by stats.py) ---
print('  CD diagrams: see results/figures/cd_diagram_mAP.pdf and cd_diagram_AP_S.pdf')

# --- 6. Per-class AP (grouped bar) ---
all_data = _read_csv('results/detection_all.csv')
CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor',
]
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLASS_NAMES))
w = 0.25
for i, neck in enumerate(NECKS):
    subset = [r for r in all_data if r['neck'] == neck]
    means = []
    for c in CLASS_NAMES:
        col = f'AP_{c}'
        vals = [float(r.get(col, 0)) for r in subset]
        means.append(np.mean(vals))
    ax.bar(x + i * w, means, w, color=COLORS[neck], label=NECK_LABELS[neck])
ax.set_xticks(x + w)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('AP'); ax.legend(); ax.set_title('Per-Class AP')
_save(fig, 'per_class_ap')

# --- 7. Training time (bar chart) ---
fig, ax = plt.subplots(figsize=(5, 4))
for i, neck in enumerate(NECKS):
    subset = [r for r in all_data if r['neck'] == neck]
    times = [float(r.get('train_time_h', 0)) for r in subset]
    ax.bar(i, np.mean(times), color=COLORS[neck], label=NECK_LABELS[neck],
           yerr=np.std(times), capsize=4)
ax.set_xticks(range(3)); ax.set_xticklabels([NECK_LABELS[n] for n in NECKS])
ax.set_ylabel('Training Time (h)'); ax.set_title('Mean Training Time (24 epochs)')
_save(fig, 'training_time')

# --- 8. Convergence curves (val_mAP + val_AP_S) ---
conv_sum = _read_csv('results/convergence_summary.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for neck in NECKS:
    rows = sorted([r for r in conv_sum if r['neck'] == neck],
                  key=lambda r: int(r['epoch']))
    epochs = [int(r['epoch']) for r in rows]
    for ax, metric, title in [(ax1, 'val_mAP', 'val mAP'),
                               (ax2, 'val_AP_S', 'val AP_S')]:
        means = [float(r.get(f'{metric}_mean') or 0) for r in rows]
        stds = [float(r.get(f'{metric}_std') or 0) for r in rows]
        means = np.array(means); stds = np.array(stds)
        ax.plot(epochs, means, 'o-', color=COLORS[neck],
                label=NECK_LABELS[neck], markersize=3)
        ax.fill_between(epochs, means - stds, means + stds,
                        color=COLORS[neck], alpha=0.15)
        ax.set_xlabel('Epoch'); ax.set_ylabel(title)
        ax.set_title(title); ax.legend()
_save(fig, 'convergence_curves')

# --- 9. Loss curves (train_loss) ---
fig, ax = plt.subplots(figsize=(6, 4))
for neck in NECKS:
    rows = sorted([r for r in conv_sum if r['neck'] == neck],
                  key=lambda r: int(r['epoch']))
    epochs = [int(r['epoch']) for r in rows]
    means = [float(r.get('train_loss_mean') or 0) for r in rows]
    ax.plot(epochs, means, 'o-', color=COLORS[neck],
            label=NECK_LABELS[neck], markersize=3)
ax.set_xlabel('Epoch'); ax.set_ylabel('Train Loss')
ax.legend(); ax.set_title('Training Loss Convergence')
_save(fig, 'loss_curves')

# --- 10. Parameter count (bar: neck-only vs total) ---
params = _read_csv('results/neck_params.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
for i, neck in enumerate(NECKS):
    row = [r for r in params if r['neck'] == neck][0]
    ax1.bar(i, int(row['params_neck']) / 1e6, color=COLORS[neck])
    ax2.bar(i, int(row['params_total']) / 1e6, color=COLORS[neck])
for ax, title in [(ax1, 'Neck Parameters'), (ax2, 'Total Parameters')]:
    ax.set_xticks(range(3))
    ax.set_xticklabels([NECK_LABELS[n] for n in NECKS])
    ax.set_ylabel('Parameters (M)'); ax.set_title(title)
_save(fig, 'parameter_count')

# --- 11. Qualitative examples note ---
print('  Qualitative images: results/figures/qualitative/')
print('\nAll figures saved to results/figures/')